# LLM に訂正の良し悪しを判定させて人手評価と比べる (メタ評価)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/26_gec_education/gec_llm_evaluator.ipynb)

NLP クックブック 26-07 ／ 文法誤り訂正と言語教育

複数の訂正システムの出力のうちどれが良いかを LLM に判定させたい。その判定が人の評価とどれくらい合うかを確かめる (評価指標の評価 = メタ評価)。

In [1]:
import itertools
import numpy as np
import torch
from scipy.stats import kendalltau, spearmanr
from transformers import AutoModelForCausalLM, AutoTokenizer

name = "sbintuitions/sarashina2.2-3b-instruct-v0.1"
tokenizer = AutoTokenizer.from_pretrained(name)
llm = AutoModelForCausalLM.from_pretrained(name).eval()
AB = [tokenizer(c, add_special_tokens=False)["input_ids"][-1] for c in ["A", "B"]]

@torch.no_grad()
def prefer_a(source, a, b):
    prompt = ("日本語学習者が書いた「元の文」に対する 2 つの訂正 A と B があります。文法的に正しく、自然で、元の文の意味を変えていないほうを選び、"
              f"A か B の 1 文字だけで答えてください。\n\n元の文: {source}\n訂正 A: {a}\n訂正 B: {b}")
    text = tokenizer.apply_chat_template([{"role": "user", "content": prompt}], add_generation_prompt=True, tokenize=False)
    ids = tokenizer(text, add_special_tokens=False, return_tensors="pt")["input_ids"]
    return float(llm(ids).logits[0, -1, AB].softmax(-1)[0])

def compare(source, x, y):
    """提示順を入れ替えて 2 回聞き、平均する (x のほうが良い確率)"""
    return (prefer_a(source, x, y) + 1 - prefer_a(source, y, x)) / 2

In [2]:
# 元の文ごとに、質の違う 4 つの訂正と、人が付けた順位 (1 が最良。同順位あり) を用意する
data = [
    ("私は毎日学校を行きます。", [("私は毎日学校に行きます。", 1), ("私は毎日学校へ通っています。", 2), ("私は毎日学校を行きます。", 3), ("私は毎日会社に行きます。", 4)]),
    ("昨日は友達と映画を見ます。", [("昨日は友達と映画を見ました。", 1), ("昨日、友人と一緒に映画を鑑賞しました。", 2), ("昨日は友達と映画を見ます。", 3), ("昨日は友達に映画が見ました。", 4)]),
    ("この料理はとてもおいしいでした。", [("この料理はとてもおいしかったです。", 1), ("この料理はとてもおいしいです。", 2), ("この料理はとてもおいしいでした。", 3), ("この料理はとてもまずかったです。", 4)]),
    ("図書館で本が借りました。", [("図書館で本を借りました。", 1), ("図書館から本を借りてきました。", 2), ("図書館で本が借りました。", 3), ("図書館に本を貸しました。", 4)]),
    ("日本に来たばかりので、日本語がまだ下手です。", [("日本に来たばかりなので、日本語がまだ下手です。", 1), ("来日して間もないため、日本語はまだ上手ではありません。", 2),
                                                 ("日本に来たばかりので、日本語がまだ下手です。", 3), ("日本に来たばかりなので、日本語がもう上手です。", 4)]),
    ("彼は私にプレゼントをもらいました。", [("彼は私にプレゼントをくれました。", 1), ("私は彼にプレゼントをもらいました。", 1), ("彼は私にプレゼントをもらいました。", 3), ("彼は私のプレゼントを捨てました。", 4)]),
]

sentence_tau, all_llm, all_human = [], [], []
for source, candidates in data:
    texts, human = [c for c, _ in candidates], [r for _, r in candidates]
    wins = np.zeros(len(texts))
    for i, j in itertools.combinations(range(len(texts)), 2):
        p = compare(source, texts[i], texts[j])
        wins[i] += p; wins[j] += 1 - p               # 総当たりの「勝ち数」(確率の和) をスコアにする
    tau = kendalltau(-wins, human).statistic
    sentence_tau.append(tau); all_llm += list(-wins); all_human += human
    print(f"{source}  τ = {tau:.2f}")
    for k in np.argsort(-wins):
        print(f"    LLM {wins[k]:.2f} / 人の順位 {human[k]} | {texts[k]}")
print("文ごとの Kendall τ の平均:", round(float(np.mean(sentence_tau)), 3), "/ 全体の Spearman:", round(spearmanr(all_llm, all_human).statistic, 3))

私は毎日学校を行きます。  τ = 0.67
    LLM 1.97 / 人の順位 2 | 私は毎日学校へ通っています。
    LLM 1.96 / 人の順位 1 | 私は毎日学校に行きます。
    LLM 1.17 / 人の順位 3 | 私は毎日学校を行きます。
    LLM 0.90 / 人の順位 4 | 私は毎日会社に行きます。
昨日は友達と映画を見ます。  τ = 0.00
    LLM 1.77 / 人の順位 3 | 昨日は友達と映画を見ます。
    LLM 1.71 / 人の順位 2 | 昨日、友人と一緒に映画を鑑賞しました。
    LLM 1.60 / 人の順位 1 | 昨日は友達と映画を見ました。
    LLM 0.92 / 人の順位 4 | 昨日は友達に映画が見ました。
この料理はとてもおいしいでした。  τ = 0.67
    LLM 1.94 / 人の順位 2 | この料理はとてもおいしいです。
    LLM 1.82 / 人の順位 1 | この料理はとてもおいしかったです。
    LLM 1.82 / 人の順位 3 | この料理はとてもおいしいでした。
    LLM 0.42 / 人の順位 4 | この料理はとてもまずかったです。
図書館で本が借りました。  τ = 1.00
    LLM 2.03 / 人の順位 1 | 図書館で本を借りました。
    LLM 1.93 / 人の順位 2 | 図書館から本を借りてきました。
    LLM 1.43 / 人の順位 3 | 図書館で本が借りました。
    LLM 0.62 / 人の順位 4 | 図書館に本を貸しました。
日本に来たばかりので、日本語がまだ下手です。  τ = 0.67
    LLM 2.61 / 人の順位 1 | 日本に来たばかりなので、日本語がまだ下手です。
    LLM 1.34 / 人の順位 3 | 日本に来たばかりので、日本語がまだ下手です。
    LLM 1.18 / 人の順位 2 | 来日して間もないため、日本語はまだ上手ではありません。
    LLM 0.87 / 人の順位 4 | 日本に来たばかりなので、日本語がもう上手です。
彼は私にプレゼントをもらいました。  τ = 0.18
    LLM 2.36 / 人の順位 3 

In [3]:
# システム単位のメタ評価: 訂正の種類を「システム」と見なし、全文での平均スコアの順位が人の順位と合うか
systems = ["最小限の正しい訂正", "流暢な書き直し", "訂正なし", "意味を変えた訂正"]
llm_mean = -np.array(all_llm).reshape(len(data), 4).mean(axis=0)
human_mean = np.array(all_human).reshape(len(data), 4).mean(axis=0)
for k in np.argsort(-llm_mean):
    print(f"{systems[k]}: LLM の平均勝ち数 {llm_mean[k]:.2f} / 人の平均順位 {human_mean[k]:.2f}")
print("システム単位の Spearman:", round(spearmanr(-llm_mean, human_mean).statistic, 3))

最小限の正しい訂正: LLM の平均勝ち数 1.99 / 人の平均順位 1.00
流暢な書き直し: LLM の平均勝ち数 1.66 / 人の平均順位 1.83
訂正なし: LLM の平均勝ち数 1.65 / 人の平均順位 3.00
意味を変えた訂正: LLM の平均勝ち数 0.70 / 人の平均順位 4.00
システム単位の Spearman: 1.0


- メタ評価は「評価指標がどれくらい人の評価と合うか」を測る。文単位 (同じ元の文に対する訂正の順位) とシステム単位 (全体の平均での順位) の 2 つの粒度があり、結果が食い違うことがある。
- 総当たりの一対比較は、1〜5 の採点をさせるより小さなモデルでも安定する。ただし比較の回数は候補数の 2 乗で増える。
- 大規模な LLM による評価は、文法誤り訂正でも従来の自動評価指標より人手評価に近いと報告されている。流暢な書き直しを正当に評価できる点が、編集の一致を数える指標との違い。
- 人手評価のデータそのものにも設計上の問題がありうる (評価した時代のシステムが古い、文単位と編集単位の評価が混ざる)。メタ評価用のデータとして SEEDA が公開されている。
- ここの 6 文は手作りの動作確認用。指標を採用するかどうかは、実際のシステム出力と人手評価の組 (数百文以上) で判断する。